In [ ]:
import polars as pl
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from optbinning import BinningProcess
import lightgbm as lgb
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri('mlruns')
mlflow.set_experiment('task3')
RUN_TAG = 'mar23'

In [ ]:
it = pl.read_csv('task3/data/IT.csv')
oot = pl.read_csv('task3/data/OOT.csv')
tx = pl.read_csv('task3/data/TRANSACTIONS.csv')

it = it.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))
oot = oot.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))
tx = tx.with_columns(pl.col('TRANSACTION_TIME').str.to_datetime().alias('TX_DT'))

customers = pl.concat([
    it.select('CUSTOMER_ID', 'TIME_DT'),
    oot.select('CUSTOMER_ID', 'TIME_DT')
])
print(f'IT: {it.shape}, OOT: {oot.shape}, TX: {tx.shape}')
print(f'Customers: {customers.shape[0]}')

In [ ]:
valid_tx = customers.join(tx, on='CUSTOMER_ID', how='left')
valid_tx = valid_tx.filter(pl.col('TX_DT') < pl.col('TIME_DT'))
valid_tx = valid_tx.with_columns(
    ((pl.col('TIME_DT') - pl.col('TX_DT')).dt.total_hours() / 24).alias('days_ago')
)
print(f'Valid TX after time filter: {valid_tx.shape[0]}')

In [ ]:
def agg_features(df, suffix=''):
    return df.group_by('CUSTOMER_ID', 'TIME_DT').agg([
        pl.len().alias(f'tx_count{suffix}'),
        pl.col('TRANSACTION_AMOUNT').mean().alias(f'amt_mean{suffix}'),
        pl.col('TRANSACTION_AMOUNT').std().alias(f'amt_std{suffix}'),
        pl.col('TRANSACTION_AMOUNT').max().alias(f'amt_max{suffix}'),
        pl.col('TRANSACTION_AMOUNT').min().alias(f'amt_min{suffix}'),
        pl.col('TRANSACTION_AMOUNT').sum().alias(f'amt_sum{suffix}'),
        pl.col('TRANSACTION_AMOUNT').median().alias(f'amt_median{suffix}'),
        pl.col('TRANSACTION_FEE').mean().alias(f'fee_mean{suffix}'),
        pl.col('TRANSACTION_FEE').std().alias(f'fee_std{suffix}'),
        pl.col('TRANSACTION_FEE').sum().alias(f'fee_sum{suffix}'),
        pl.col('days_ago').mean().alias(f'days_ago_mean{suffix}'),
        pl.col('days_ago').min().alias(f'days_ago_min{suffix}'),
        pl.col('days_ago').std().alias(f'days_ago_std{suffix}'),
        (pl.col('TRANSACTION_TYPE') == 'CREDIT').sum().alias(f'n_credit{suffix}'),
        (pl.col('TRANSACTION_TYPE') == 'DEBIT').sum().alias(f'n_debit{suffix}'),
        pl.col('TRANSACTION_CLASS').n_unique().alias(f'n_classes{suffix}'),
        pl.col('TRANSACTION_PLACE').n_unique().alias(f'n_places{suffix}'),
        pl.col('TRANSACTION_PURPOSE').n_unique().alias(f'n_purposes{suffix}'),
    ])

feat_all = agg_features(valid_tx, '')

windows = [7, 14, 30, 90, 180]
feat_windows = [feat_all]
for w in windows:
    sub = valid_tx.filter(pl.col('days_ago') <= w)
    feat_windows.append(agg_features(sub, f'_{w}d'))

features = feat_windows[0]
for fw in feat_windows[1:]:
    features = features.join(fw, on=['CUSTOMER_ID', 'TIME_DT'], how='left')

print(f'Features shape: {features.shape}')

In [ ]:
cat_cols = ['TRANSACTION_CLASS', 'TRANSACTION_TYPE', 'TRANSACTION_PURPOSE', 'TRANSACTION_PLACE']
cat_pivot_parts = []

for col in cat_cols:
    pivot = (
        valid_tx.group_by('CUSTOMER_ID', 'TIME_DT', col)
        .agg(pl.len().alias('cnt'))
        .pivot(on=col, index=['CUSTOMER_ID', 'TIME_DT'], values='cnt')
        .fill_null(0)
    )
    pivot = pivot.rename({c: f'{col}_{c}' for c in pivot.columns if c not in ['CUSTOMER_ID', 'TIME_DT']})
    cat_pivot_parts.append(pivot)

for cp in cat_pivot_parts:
    features = features.join(cp, on=['CUSTOMER_ID', 'TIME_DT'], how='left')

features = features.fill_null(0)
print(f'Features with categoricals: {features.shape}')

In [ ]:
ratios = features.with_columns([
    (pl.col('n_credit') / (pl.col('tx_count') + 1)).alias('credit_ratio'),
    (pl.col('amt_sum') / (pl.col('tx_count') + 1)).alias('avg_tx_size'),
    (pl.col('fee_sum') / (pl.col('amt_sum').abs() + 1)).alias('fee_to_amt_ratio'),
    (pl.col('tx_count_7d') / (pl.col('tx_count') + 1)).alias('recency_ratio_7'),
    (pl.col('tx_count_14d') / (pl.col('tx_count') + 1)).alias('recency_ratio_14'),
    (pl.col('tx_count_30d') / (pl.col('tx_count') + 1)).alias('recency_ratio_30'),
    (pl.col('tx_count_90d') / (pl.col('tx_count') + 1)).alias('recency_ratio_90'),
    (pl.col('amt_sum_7d') / (pl.col('amt_sum').abs() + 1)).alias('amt_recency_7'),
    (pl.col('amt_sum_30d') / (pl.col('amt_sum').abs() + 1)).alias('amt_recency_30'),
    (pl.col('amt_sum_90d') / (pl.col('amt_sum').abs() + 1)).alias('amt_recency_90'),
    (pl.col('amt_max') - pl.col('amt_min')).alias('amt_range'),
    (pl.col('amt_std') / (pl.col('amt_mean').abs() + 1)).alias('amt_cv'),
])

features = ratios
feat_cols = [c for c in features.columns if c not in ['CUSTOMER_ID', 'TIME_DT']]
print(f'Final feature count: {len(feat_cols)}')

In [ ]:
it_feat = it.join(features, on=['CUSTOMER_ID', 'TIME_DT'], how='left').fill_null(0)
oot_feat = oot.join(features, on=['CUSTOMER_ID', 'TIME_DT'], how='left').fill_null(0)

cutoff = it_feat.sort('TIME_DT')['TIME_DT'].quantile(0.8)
train = it_feat.filter(pl.col('TIME_DT') <= cutoff)
val = it_feat.filter(pl.col('TIME_DT') > cutoff)
print(f'Train: {train.shape[0]}, Val: {val.shape[0]}')
print(f'Train target: {train["TARGET"].mean():.4f}, Val target: {val["TARGET"].mean():.4f}')

X_train = train.select(feat_cols).to_pandas()
y_train = train['TARGET'].to_numpy()
X_val = val.select(feat_cols).to_pandas()
y_val = val['TARGET'].to_numpy()
X_oot = oot_feat.select(feat_cols).to_pandas()

In [ ]:
bp = BinningProcess(
    variable_names=feat_cols,
    min_prebin_size=0.02, min_bin_size=0.05, max_n_bins=10,
    selection_criteria={'iv': {'min': 0.02}}
)
bp.fit(X_train.values, y_train)
selected = list(bp.get_support(names=True))
print(f'Selected (IV>=0.02): {len(selected)}')

X_tr_woe = bp.transform(X_train.values, metric='woe')
X_va_woe = bp.transform(X_val.values, metric='woe')
X_oo_woe = bp.transform(X_oot.values, metric='woe')

best_gini_lr = 0
best_lr = None
for C in [0.01, 0.1, 0.5, 1.0]:
    lr = LogisticRegression(C=C, max_iter=1000, solver='lbfgs')
    lr.fit(X_tr_woe, y_train)
    g = 2 * roc_auc_score(y_val, lr.predict_proba(X_va_woe)[:, 1]) - 1
    print(f'WOE+LR C={C}: val_gini={g:.4f}')
    if g > best_gini_lr:
        best_gini_lr, best_lr = g, lr

p_lr_val = best_lr.predict_proba(X_va_woe)[:, 1]
print(f'Best WOE+LR: val_gini={best_gini_lr:.4f}')

In [ ]:
configs = [
    {'num_leaves': 15, 'learning_rate': 0.01, 'min_child_samples': 300, 'feature_fraction': 0.6, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 1.0, 'reg_lambda': 1.0},
    {'num_leaves': 31, 'learning_rate': 0.01, 'min_child_samples': 200, 'feature_fraction': 0.7, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 0.5, 'reg_lambda': 0.5},
    {'num_leaves': 15, 'learning_rate': 0.05, 'min_child_samples': 200, 'feature_fraction': 0.9, 'bagging_fraction': 0.9, 'bagging_freq': 3, 'reg_alpha': 1.0, 'reg_lambda': 1.0},
    {'num_leaves': 7, 'learning_rate': 0.01, 'min_child_samples': 500, 'feature_fraction': 0.5, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 2.0, 'reg_lambda': 2.0},
    {'num_leaves': 7, 'learning_rate': 0.05, 'min_child_samples': 300, 'feature_fraction': 0.6, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'reg_alpha': 1.0, 'reg_lambda': 1.0},
]

best_lgb_gini = 0
model = None

for i, cfg in enumerate(configs):
    params = {'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'feature_pre_filter': False, **cfg}
    lgb_tr = lgb.Dataset(X_train, y_train, free_raw_data=False)
    lgb_va = lgb.Dataset(X_val, y_val, reference=lgb_tr, free_raw_data=False)
    m = lgb.train(params, lgb_tr, num_boost_round=3000, valid_sets=[lgb_va],
                  callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    p = m.predict(X_val)
    g = 2 * roc_auc_score(y_val, p) - 1
    print(f'Config {i}: val_gini={g:.4f} (leaves={cfg["num_leaves"]}, lr={cfg["learning_rate"]}, iters={m.best_iteration})')
    if g > best_lgb_gini:
        best_lgb_gini, model = g, m

p_lgb_val = model.predict(X_val)
print(f'\nBest LGB: val_gini={best_lgb_gini:.4f}')

In [ ]:
best_ens_g = 0
best_w = 0
for w in np.arange(0.0, 1.05, 0.05):
    p_ens = w * p_lr_val + (1 - w) * p_lgb_val
    g = 2 * roc_auc_score(y_val, p_ens) - 1
    if g > best_ens_g:
        best_ens_g, best_w = g, w

print(f'Best ensemble: w_lr={best_w:.2f}, val_gini={best_ens_g:.4f}')

results = {'WOE_LR': best_gini_lr, 'LGB': best_lgb_gini, 'Ensemble': best_ens_g}
best_name = max(results, key=results.get)
val_gini = results[best_name]
print(f'Best: {best_name} = {val_gini:.4f}')
for k, v in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {k}: {v:.4f}')

In [ ]:
if best_name == 'WOE_LR':
    p_oot_final = best_lr.predict_proba(X_oo_woe)[:, 1]
elif best_name == 'LGB':
    p_oot_final = model.predict(X_oot)
else:
    p_oot_final = best_w * best_lr.predict_proba(X_oo_woe)[:, 1] + (1 - best_w) * model.predict(X_oot)

preds = pl.DataFrame({'CUSTOMER_ID': oot_feat['CUSTOMER_ID'], 'SCORE': p_oot_final})
preds.write_csv('task3/predictions.csv')
print(f'OOT predictions: {preds.shape}')

with mlflow.start_run(run_name='v3_more_windows'):
    mlflow.log_param('model', best_name)
    mlflow.log_param('n_features', len(feat_cols))
    mlflow.log_param('n_selected_woe', len(selected))
    mlflow.log_metric('val_gini', val_gini)
    mlflow.log_metric('val_gini_lr', best_gini_lr)
    mlflow.log_metric('val_gini_lgb', best_lgb_gini)
    mlflow.log_metric('val_gini_ens', best_ens_g)
    mlflow.set_tag('task', 'task3')
    mlflow.set_tag('run_tag', RUN_TAG)
print(f'val_gini: {val_gini:.4f}')